# Gradient Boosting 시계열 예측

## 철강산업 전력사용량의 15분 선행 예측

7교시의 시간순 검증과 persistence baseline을 유지한 채 Gradient Boosting 모형을 추가해 복잡한 모형의 실질적 개선을 평가합니다.


## 이번 교시의 학습목표

수업이 끝나면 다음을 수행할 수 있습니다.

- Gradient Boosting이 이전 모형의 손실을 순차적으로 보완하는 원리를 설명합니다.
- learning rate, tree 복잡도와 iteration 수의 관계를 설명합니다.
- boosting과 persistence를 동일한 미래 test 구간에서 비교하고 개선 범위를 기록합니다.


## 이번 교시에서 사용할 데이터

### 강의 시연: Steel 15분·24시간 lag 자료

7교시와 같은 Steel 시계열에서 15분 전과 24시간 전 전력사용량을 feature로 만듭니다. 원자료는 35,040행이지만 lag가 존재하지 않는 초기·불연속 관측은 모형 입력에서 제외되므로 실제 학습행 수가 줄어듭니다.

- **target:** 현재 시점의 `Usage_kWh`
- **주요 예측정보:** 15분 전 사용량, 하루 전 같은 시각 사용량, 달력·부하조건
- **후보모형:** `HistGradientBoostingRegressor`
- **baseline:** 같은 미래기간의 persistence

### 적용 실습: Bike 시간별 수요

Bike 자료에서는 한 시간 전과 하루 전 수요를 이용해 같은 과거–미래 검증을 수행합니다. 날씨 열을 사용할 때는 예측시점에 실제 예보로 이용 가능한 정보인지 구분합니다. 수업 파일의 날씨 관측값을 사용한 결과를 실제 사전예측 성능으로 확대해서 말하지 않습니다.


## 알고리즘을 비교할 때 고정할 조건

연구질문은 시간순 평가설계를 유지한 상태에서 Gradient Boosting이 persistence보다
미래구간 오차를 줄이는지입니다.

| 고정할 조건 | 이유 |
|---|---|
| feature의 시간적 가용성 | 입력정보 범위를 동일하게 유지 |
| 과거–미래 경계 | 평가기간 난이도를 동일하게 유지 |
| 미래 test timestamp | 행별 오류를 직접 비교 |
| persistence와 지표 | 개선 기준과 단위를 동일하게 유지 |

모형과 평가조건을 동시에 바꾸면 점수 차이의 출처를 분리하기 어렵습니다.


## Random Forest와 Gradient Boosting의 tree 결합

| 방법 | tree 사이의 관계 | 최종 예측 |
|---|---|---|
| Random Forest | 서로 다른 표본·feature로 독립적으로 적합 | tree 예측의 평균 |
| Gradient Boosting | 현재 예측이 남긴 손실을 줄이도록 순차 적합 | 이전 예측에 보정값 누적 |

Forest는 평균화로 단일 tree의 변동을 줄이고, Boosting은 앞 단계가 남긴 오류를 다음
단계에서 보완합니다. 두 방법 모두 여러 tree를 사용하지만 학습 순서와 결합 방식은
다릅니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

forest_trees = pd.Series([90.0, 105.0, 105.0], index=["tree 1", "tree 2", "tree 3"])
forest_mean = forest_trees.mean()
boosting_path = pd.DataFrame({
    "stage": [0, 1, 2, 3],
    "prediction": [60.0, 75.0, 85.0, 90.0],
    "added_correction": [0.0, 15.0, 10.0, 5.0],
})

print("[결합방식으로 확인] Forest는 독립 예측을 평균하고 Boosting은 보정값을 누적합니다.")
display(forest_trees.rename("independent prediction").to_frame())
print("Forest mean:", forest_mean)
display(boosting_path)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].bar(forest_trees.index, forest_trees.values, color="#60a5fa")
axes[0].axhline(forest_mean, color="#dc2626", linestyle="--", label="mean")
axes[0].set(title="Random Forest: averaging", ylabel="prediction")
axes[0].legend()
axes[1].plot(boosting_path["stage"], boosting_path["prediction"], marker="o", color="#7c3aed")
axes[1].set(title="Gradient Boosting: sequential update", xlabel="stage", ylabel="prediction")
plt.tight_layout()
plt.show()


## Gradient Boosting의 순차 잔차 보정

회귀의 제곱오차 예에서는 현재 예측의 잔차를 다음 작은 tree가 근사합니다.

$$F_m(x)=F_{m-1}(x)+\eta h_m(x)$$

- $F_{m-1}$: 현재까지 결합한 예측
- $h_m$: 남은 오류를 줄이는 방향을 근사한 tree
- $\eta$: 한 단계의 보정폭을 조절하는 learning rate

실제값 `40, 60, 80`을 모두 60으로 시작하면 잔차는 `-20, 0, +20`입니다. 다음
tree가 그 일부를 보정하면 예측은 실제값 방향으로 이동합니다.

출처: [Friedman (2001)](https://doi.org/10.1214/aos/1013203451)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

update = pd.DataFrame({
    "row": ["A", "B", "C"],
    "actual": [40.0, 60.0, 80.0],
    "current_prediction": [60.0, 60.0, 60.0],
})
update["residual"] = update["actual"] - update["current_prediction"]
update["next_tree_correction"] = 0.5 * update["residual"]
update["updated_prediction"] = update["current_prediction"] + update["next_tree_correction"]
update["absolute_error_before"] = (update["actual"] - update["current_prediction"]).abs()
update["absolute_error_after"] = (update["actual"] - update["updated_prediction"]).abs()

print("[잔차로 확인] 다음 tree의 보정이 현재 예측을 실제값 방향으로 이동시킵니다.")
display(update)
print("이 표는 계산 흐름을 보이기 위해 각 행 잔차의 50%를 정확히 보정한 예시입니다.")
print("실제 boosting tree는 여러 행의 잔차 패턴을 leaf 단위로 근사하므로 각 행 잔차의 일정 비율을 정확히 재현하지 않습니다.")

fig, ax = plt.subplots(figsize=(7, 3.7))
x = range(len(update))
ax.plot(x, update["actual"], marker="o", label="actual")
ax.plot(x, update["current_prediction"], marker="o", label="before update")
ax.plot(x, update["updated_prediction"], marker="o", label="after update")
ax.set(xticks=list(x), xticklabels=update["row"], ylabel="target", title="One residual-correction stage")
ax.legend()
plt.tight_layout()
plt.show()


## Learning rate와 반복 횟수의 경로

learning rate가 작으면 한 단계의 이동폭이 작아 보통 더 많은 반복이 필요합니다.
값이 크면 빠르게 이동하지만 제한된 validation 표본의 패턴을 지나치게 따라갈 수
있습니다.

learning rate, tree의 leaf 수·깊이와 iteration 수는 함께 작동합니다. 이 조합은
train 내부의 시간순 validation에서 정하고, 최종 test는 설정을 고정한 뒤 한 번만
사용합니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

target = 80.0
initial_prediction = 60.0
iterations = 6
paths = []
for learning_rate in [0.10, 0.30, 0.70]:
    prediction = initial_prediction
    paths.append({"learning_rate": learning_rate, "stage": 0, "prediction": prediction})
    for stage in range(1, iterations + 1):
        residual = target - prediction
        prediction = prediction + learning_rate * residual
        paths.append({"learning_rate": learning_rate, "stage": stage, "prediction": prediction})

path_table = pd.DataFrame(paths)
final_table = path_table.groupby("learning_rate", as_index=False).tail(1)
print("[경로로 확인] learning rate는 한 단계에서 잔차의 얼마를 보정할지 정합니다.")
display(final_table.round(2))

fig, ax = plt.subplots(figsize=(7, 3.8))
for learning_rate, group in path_table.groupby("learning_rate"):
    ax.plot(group["stage"], group["prediction"], marker="o", label=f"eta={learning_rate}")
ax.axhline(target, color="black", linestyle="--", label="target")
ax.set(xlabel="boosting stage", ylabel="prediction", title="Learning-rate paths toward the same target")
ax.legend()
plt.tight_layout()
plt.show()


## Lag와 날씨정보의 이용 가능 시점

| feature | 기준시각 | forecast origin에서 사용 가능 조건 |
|---|---|---|
| 단기 lag | 15분 또는 1시간 전 | 과거 target 저장이 완료됨 |
| 일주기 lag | 정확히 24시간 전 | 해당 timestamp가 존재함 |
| 달력정보 | 미래 시각의 시각·요일 | 사전에 계산 가능 |
| target-time 실제 날씨 | 예측 대상 시각 | origin 이후 관측이면 사용 불가 |
| 날씨예보 | 예보 발행시각 | origin 이전에 발행된 예보만 사용 가능 |

결과시각의 실제 날씨를 사용한 분석은 조건부 benchmark일 수 있습니다. 운영 가능한
미래예측으로 해석하려면 당시 제공된 예보와 예보오차로 다시 평가합니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

origin = pd.Timestamp("2026-01-02 10:00")
target_time = origin + pd.Timedelta(minutes=15)
availability = pd.DataFrame({
    "feature": ["lag_15m", "lag_24h", "calendar_hour", "observed_weather_at_target", "weather_forecast_for_target"],
    "value_time": [
        target_time - pd.Timedelta(minutes=15),
        target_time - pd.Timedelta(hours=24),
        target_time,
        target_time,
        target_time,
    ],
    "available_at": [
        origin,
        target_time - pd.Timedelta(hours=24),
        origin - pd.Timedelta(days=1),
        target_time + pd.Timedelta(minutes=5),
        origin - pd.Timedelta(hours=1),
    ],
})
availability["available_at_origin"] = availability["available_at"].le(origin)

print("[정보시점으로 확인] 값의 기준시각과 실제 이용 가능 시각은 다를 수 있습니다.")
print("forecast origin:", origin, "| target time:", target_time)
display(availability)

fig, ax = plt.subplots(figsize=(9, 4.2))
colors = availability["available_at_origin"].map({True: "#2563eb", False: "#dc2626"})
ax.scatter(availability["available_at"], availability["feature"], c=colors, s=75)
ax.axvline(origin, color="black", linestyle="--", label="forecast origin")
ax.set(xlabel="time when the value becomes available", title="Target-time observed weather is unavailable at the origin")
ax.legend()
plt.tight_layout()
plt.show()


## HistGradientBoostingRegressor와 결과 보고

`HistGradientBoostingRegressor`는 연속형 feature를 bin으로 요약해 histogram 기반으로
분기 후보를 계산하는 scikit-learn의 CPU 모형입니다. GPU와 TensorFlow는 필요하지
않습니다.

결과에는 chronological test 기간, persistence와 boosting의 동일 timestamp, MAE·RMSE
단위, 기준모형 대비 개선량과 큰 잔차 구간을 기록합니다. 한 번의 holdout 결과를 다른
계절·연도·공정으로 일반화하려면 rolling-origin과 외부표본 평가가 추가로 필요합니다.


## 실행 환경과 입력자료

7교시와 같은 Steel 시간순 평가조건을 유지하고 후보모형만 추가합니다.

| 확인 항목 | 이번 분석의 기준 |
|---|---|
| target | `Usage_kWh` |
| 입력정보 | 과거 lag와 사전에 정한 달력정보 |
| 기준모형 | persistence |
| 후보모형 | 고정 설정의 HistGradientBoosting |
| 비교조건 | 동일한 미래 timestamp의 MAE·RMSE |

다음 셀에서 lag 생성 전후 행 수, 학습 종료시각, 평가 시작시각과 첫 예측행을 확인합니다.

> **실행 전 확인:** 복잡한 후보의 추가가치를 판단하려면 어떤 행과 지표를 기준모형과 같게 유지해야 합니까?


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행하십시오."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 1. 예측 시점에서 이용 가능한 변수

| feature | 기준 시각 | 사용 조건 |
|---|---|---|
| `lag_15m` | 현재보다 15분 전 | 직전 측정 완료 |
| `lag_24h` | 현재보다 24시간 전 | 하루 전 기록 존재 |
| `hour_sin`, `hour_cos`, `NSM` | 현재 달력·시각 | 사전 계산 가능 |
| 날씨·운전조건 | 예보·계획 또는 실측 시각 | 운영 시 가용성 별도 명시 |

두 lag는 timestamp join으로 생성합니다. 정확한 과거 짝이 없는 초기·불연속 행은 모델 입력에서 제외되므로 원자료와 모델자료의 행 수를 모두 기록합니다.

> **확인:** 실제 날씨 관측값을 사전예측 변수로 사용할 때 어떤 예보 가정을 추가해야 할까요?


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

raw = pd.read_csv(find_course_data("steel/Steel_industry_data.csv"))
raw["timestamp"] = pd.to_datetime(raw["date"], dayfirst=True)
raw = raw.sort_values("timestamp")
target_name = "Usage_kWh"
analysis_name = "Steel gradient boosting forecast"

def attach_lag(frame, hours, name):
    lag = frame[["timestamp", target_name]].copy()
    lag["timestamp"] = lag["timestamp"] + pd.Timedelta(hours=hours)
    return frame.merge(lag.rename(columns={target_name: name}), on="timestamp", how="left")

data = attach_lag(raw, 0.25, "lag_15m")
data = attach_lag(data, 24, "lag_24h").dropna(subset=["lag_15m", "lag_24h"]).copy()
data["hour_sin"] = np.sin(2*np.pi*data["timestamp"].dt.hour/24)
data["hour_cos"] = np.cos(2*np.pi*data["timestamp"].dt.hour/24)
features = ["lag_15m", "lag_24h", "hour_sin", "hour_cos", "NSM"]
baseline_feature = "lag_15m"

data = data.sort_values("timestamp").reset_index(drop=True)
print("model rows:", len(data))
print("features:", features)
display(data[["timestamp", target_name, *features]].head(3))


### 출력 해석: 두 lag의 시간 의미

현재 사용량과 15분 전·24시간 전 값을 한 행에서 비교합니다.


In [ ]:
lag_feature_view = data[["timestamp", target_name, "lag_15m", "lag_24h", "NSM"]].head(8).copy()
print("lag_15m는 직전 관측, lag_24h는 하루 전 같은 시각의 관측입니다.")
display(lag_feature_view)


## 2. 순차 보정과 시간순 학습

Gradient Boosting은 현재 결합모형의 손실을 줄이는 tree를 순차적으로 추가합니다.

| 수치 예 | 값 |
|---|---|
| 실제값 | `[40, 60, 80]` |
| 초기예측 | `[60, 60, 60]` |
| 잔차 | `[-20, 0, +20]` |
| 보정값 | `[-10, 0, +10]` |
| 갱신예측 | `[50, 60, 70]` |

실제 적합에서는 과거 80%만 `fit`에 사용하며 미래 20%는 평가에 남겨 둡니다. 코드 셀은 모형을 적합하므로 출력이 없는 것이 정상이고, 다음 셀에서 수치 예와 실제 train/test 기간을 확인합니다.

> **확인:** Random Forest의 평균화와 Boosting의 순차 보정은 tree 사이의 관계가 어떻게 다를까요?


In [ ]:
cut = int(len(data) * 0.80)
train, test = data.iloc[:cut], data.iloc[cut:]
X_train, y_train = train[features], train[target_name]
X_test, y_test = test[features], test[target_name]

model = HistGradientBoostingRegressor(
    learning_rate=0.06, max_iter=160, max_leaf_nodes=31,
    l2_regularization=0.1, random_state=42,
)
model.fit(X_train, y_train)
prediction = model.predict(X_test)
baseline = X_test[baseline_feature].to_numpy()


### 출력 해석: 작은 잔차 보정과 실제 학습 기간

세 숫자로 boosting의 한 단계를 계산한 뒤 모델이 본 마지막 시각과 평가가 시작되는 첫 시각을 확인합니다.


In [ ]:
toy_boosting = pd.DataFrame({
    "actual": [40.0, 60.0, 80.0],
    "first prediction": [60.0, 60.0, 60.0],
})
toy_boosting["residual"] = toy_boosting["actual"] - toy_boosting["first prediction"]
toy_boosting["next tree correction"] = 0.5 * toy_boosting["residual"]
toy_boosting["updated prediction"] = (
    toy_boosting["first prediction"] + toy_boosting["next tree correction"]
)
print("[한 단계의 순차적 보정]")
display(toy_boosting)

print("train:", train["timestamp"].min(), "→", train["timestamp"].max(), "| rows:", len(train))
print("test :", test["timestamp"].min(), "→", test["timestamp"].max(), "| rows:", len(test))
boosting_preview = pd.DataFrame({
    "timestamp": test["timestamp"].iloc[:10].to_numpy(),
    "observed": y_test.iloc[:10].to_numpy(),
    "persistence": baseline[:10],
    "boosting": prediction[:10],
})
display(boosting_preview)


## 3. Persistence 대비 예측 성능

| 비교 항목 | 기준 |
|---|---|
| 평가행 | 동일한 미래 timestamp |
| horizon | 15분 선행 |
| 기준모형 | `lag_15m` persistence |
| 지표 | MAE·RMSE, kWh |
| 행별 근거 | 실제값·두 예측·두 절대오차 |
| 시간축 근거 | 실제값·persistence·boosting의 동일 구간 |

첫 8개 미래행의 오차를 확인한 뒤 전체 test 지표를 읽습니다. 평균오차가 감소해도 피크 구간의 과소예측이 남을 수 있으므로 시간축 그림과 큰 잔차를 병행해 해석합니다.

> **확인:** MAE는 감소했지만 RMSE가 비슷하다면 어떤 오류가 남아 있을 가능성이 있을까요?


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

result_table = pd.DataFrame([
    {"model": "persistence", "MAE": mean_absolute_error(y_test, baseline), "RMSE": root_mean_squared_error(y_test, baseline)},
    {"model": "hist gradient boosting", "MAE": mean_absolute_error(y_test, prediction), "RMSE": root_mean_squared_error(y_test, prediction)},
])
result_by_model = result_table.set_index("model")
baseline_mae = float(result_by_model.loc["persistence", "MAE"])
candidate_mae = float(result_by_model.loc["hist gradient boosting", "MAE"])
baseline_rmse = float(result_by_model.loc["persistence", "RMSE"])
candidate_rmse = float(result_by_model.loc["hist gradient boosting", "RMSE"])
mae_difference = baseline_mae - candidate_mae
rmse_difference = baseline_rmse - candidate_rmse
mae_direction = "낮았습니다" if mae_difference >= 0 else "높았습니다"
rmse_direction = "낮았습니다" if rmse_difference >= 0 else "높았습니다"
evaluation_period = f"{test['timestamp'].min():%Y-%m-%d %H:%M}–{test['timestamp'].max():%Y-%m-%d %H:%M}"
metric_unit = 'kWh'
data_scope = 'Steel 15분 간격 전력사용 자료'
interpretation_scope = (
    WEATHER_BENCHMARK_NOTE
    if False and "WEATHER_BENCHMARK_NOTE" in globals() and TARGET_TIME_WEATHER_USED
    else "한 번의 시간순 평가기간 결과이며 다른 기간의 성능은 추가 검증이 필요합니다."
)
result_claim = (
    f"{data_scope}의 {evaluation_period} 시간순 test에서 HistGradientBoosting의 MAE는 "
    f"{candidate_mae:.2f} {metric_unit}였으며, persistence의 {baseline_mae:.2f} {metric_unit}보다 "
    f"{abs(mae_difference):.2f} {metric_unit} {mae_direction}. "
    f"RMSE는 {candidate_rmse:.2f} {metric_unit}였으며, persistence의 {baseline_rmse:.2f} {metric_unit}보다 "
    f"{abs(rmse_difference):.2f} {metric_unit} {rmse_direction}. {interpretation_scope}"
)
print("\n[평가자료에서 자동으로 만든 결과문장]")
print(result_claim)

display(result_table)

view = min(200, len(test))
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(test["timestamp"].iloc[:view], y_test.iloc[:view], label="observed", linewidth=1.8)
ax.plot(test["timestamp"].iloc[:view], baseline[:view], label="persistence", linewidth=1.3)
ax.plot(test["timestamp"].iloc[:view], prediction[:view], label="boosting", linewidth=1.3)
ax.set(title="Chronological test: observed vs prediction", xlabel="Time", ylabel=target_name)
ax.legend()
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


### 출력 해석: 같은 미래 행의 오류와 전체 지표

첫 미래 행들에서 persistence와 boosting의 절대오차를 직접 계산하고, 같은 test 전체의 MAE·RMSE와 연결합니다.


In [ ]:
boosting_error_view = pd.DataFrame({
    "timestamp": test["timestamp"].iloc[:8].to_numpy(),
    "observed": y_test.iloc[:8].to_numpy(),
    "persistence": baseline[:8],
    "boosting": prediction[:8],
})
boosting_error_view["persistence_abs_error"] = (
    boosting_error_view["observed"] - boosting_error_view["persistence"]
).abs()
boosting_error_view["boosting_abs_error"] = (
    boosting_error_view["observed"] - boosting_error_view["boosting"]
).abs()
print("[같은 미래 여덟 행의 실제값·예측값·절대오차]")
display(boosting_error_view.round(3))
first_error = boosting_error_view.iloc[0]
print(
    f"첫 미래행 절대오차: persistence {first_error['persistence_abs_error']:.3f} kWh, "
    f"boosting {first_error['boosting_abs_error']:.3f} kWh"
)
print("[같은 미래 test 전체의 MAE와 RMSE]")
display(result_table.round(3))


## 4. Boosting 결과 검증

| 항목 | 통과 조건 |
|---|---|
| lag | 과거 timestamp에서만 생성 |
| 분할 | train과 test 기간이 겹치지 않음 |
| 입력·예측 | 결측·무한값 없음 |
| 비교 | persistence와 boosting의 test 행·길이 일치 |
| 보고 | MAE·RMSE, 개선량, 평가기간 명시 |
| 오류 | 피크·큰 잔차를 별도로 확인 |

검증 범위는 선택한 미래 holdout입니다. 다른 계절·설비·forecast horizon과 계산비용은 추가 평가 항목입니다.


In [ ]:
checks = pd.Series({
    "result claim contains evaluated evidence": evaluation_period in result_claim and metric_unit in result_claim and all(f"{value:.2f}" in result_claim for value in [baseline_mae, candidate_mae, baseline_rmse, candidate_rmse]),
    "result scope recorded": len(interpretation_scope.strip()) >= 30,
    "strict time order": train["timestamp"].max() < test["timestamp"].min(),
    "no feature missing": not data[features].isna().any().any(),
    "finite prediction": np.isfinite(prediction).all(),
    "same test length": len(prediction) == len(y_test),
    "baseline comparison complete": set(result_table["model"]) == {"persistence", "hist gradient boosting"},
}, name="passed")
display(checks.to_frame())
assert checks.all()
print("시계열 예측 비교 검증: PASS")


## ChatGPT 저장 응답 검증: Boosting의 추가가치 확인하기

모형 이름이나 복잡도보다 **같은 미래기간·예측간격·입력정보·평가지표에서 직전 값 기준모형보다 오차가 줄었는지**를 검토합니다.

답변이 평가자료를 보며 설정을 다시 고르도록 제안하거나 한 번의 미래구간 결과를 모든 계절로 확대하면 채택하지 않습니다.


In [ ]:
review_prompt = f"""
다음 시간순 예측결과에서 Gradient Boosting의 추가가치를 검토해 주세요.

- 입력변수: {features}
- 학습 종료시각: {train['timestamp'].max()}
- 평가 시작시각: {test['timestamp'].min()}
- 결과표:\n{result_table.to_string(index=False)}\n- 코드에서 생성한 결과문장: {result_claim}

아래 형식으로 답해 주세요.
1. 기준모형과 후보모형의 동일 비교조건
2. 결과문장의 평가기간·MAE·RMSE·차이·단위가 결과표와 일치하는지
3. 학습자료 내부에서만 검토할 설정 1개
4. 다른 계절로 일반화하기 전에 필요한 검증 1개

제약: 평가자료 점수를 보며 설정을 다시 고르지 말고, 한 번의 미래기간 결과를 전체 운영기간으로 확대하지 마세요.
""".strip()
print("[ChatGPT에 복사할 프롬프트]\n")
print(review_prompt)


### ChatGPT 저장 응답을 검증할 때

1. 기준모형과 후보가 같은 미래 timestamp와 지표를 사용했는지 확인합니다.
2. 설정 변경 제안은 학습자료 내부 검증에서만 시험합니다.
3. 결과문장에는 평가기간과 아직 확인하지 않은 계절·현장을 함께 적습니다.


## 해석 범위

성능 비교는 고정된 기간·입력변수·hyperparameter와 하나의 미래 test 구간에서 수행했습니다.

다른 운영연도, 여러 forecast origin, 날씨예보 오차와 계산비용을 포함한 배포 안정성은 아직 검증하지 않았습니다. 개선 결과는 사용한 평가기간과 지표의 범위에서 보고합니다.


## 오늘의 결론과 다음 단계

### 이번 교시의 결론

같은 미래기간과 입력정보를 유지한 채 직전 값 기준모형과 Boosting을 비교하고, 설정 선택과 최종평가의 경계를 지킵니다.

### 적용 순서

1. `08_gradient_boosting_forecasting_follow.ipynb`에서 핵심 시연을 같은 조건으로 재현합니다.
2. `08_gradient_boosting_forecasting_practice.ipynb`에서 지정된 데이터 또는 조건 하나만 바꿔 비교합니다.
3. 변경조건, 비교표 또는 그림, 결과문장과 현재 검증하지 못한 범위를 함께 기록합니다.

### 실습 중 확인할 질문

- boosting은 앞선 모델의 어떤 오류를 다음 단계에서 보완합니까?
- 모델과 persistence baseline이 같은 test timestamp에서 비교됐습니까?
- 성능 개선이 어느 기간에도 유지된다고 말할 수 있습니까?

마지막 `PASS`는 Notebook에 적힌 실행조건을 확인한 결과이며 연구결론의 타당성을 대신하지 않습니다.
